In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
import itertools
from pathlib import Path
import re
from typing import Literal

from matplotlib.colors import CenteredNorm
from matplotlib import transforms
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pickle
from scipy import stats
from scipy.stats import ttest_ind, pearsonr, spearmanr
import seaborn as sns
import textgrid
import torch
from tqdm.auto import tqdm
tqdm.pandas()

from src.data import get_electrode_df, add_metadata_features
from src.models.causal4 import run_causal4_analysis
from src.models import causal4
import src.viz as viz

In [ ]:
epochs_path = "outputs/epochs_preprocessed"
tg_dir = "textgrids"

include_rois = ["superiortemporal", "postcentral", "precentral", "supramarginal", "inferiortemporal", "middletemporal",
                "parsopercularis", "caudalmiddlefrontol", "lateralorbitofrontal", "temporalpole", "inferiorparietal",
                "rostralmiddlefrontal", "parstriangularis"]

# uncorrected p-value threshold
alpha = 0.01

# ERP sources from TIMIT data
timit_epoch_sources = {
    "All": "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-all.h5",
    "Word onset": "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-onsets.h5",
}

outdir = "outputs/causal4_searchlight"

In [ ]:
all_epoch_paths = list(Path(epochs_path).glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epo", str(path))[0]
    epochs[subject_name] = mne.read_epochs(str(path))
    epochs[subject_name].metadata = add_metadata_features(epochs[subject_name].metadata)

In [ ]:
electrode_df = pd.concat([get_electrode_df(subject_name) for subject_name in epochs.keys()],
                         keys=epochs.keys(), names=["subject"])
electrode_df["roi"] = electrode_df.roi.astype(str)
electrode_df = electrode_df.droplevel("electrode_name")

# Drop electrodes metadata which don't have corresponding data
for subject, epochs_i in epochs.items():
    electrode_df.loc[subject, "keep"] = np.arange(len(electrode_df.loc[subject])) < len(epochs_i.info["ch_names"])
electrode_df = electrode_df[electrode_df["keep"]].drop(columns="keep")

electrode_df

## Find speech-responsive electrodes

In [ ]:
# power threshold relative to pre-speech baseline which defines a "speech responsive" electrode
# if we see absolute value change >= this threshold, call the electrode speech-responsive
speech_responsive_threshold = 0.3

In [ ]:
# demo this
dd = next(iter(epochs.values())).copy().apply_baseline((-0.1, 0)).average().crop(tmin=0, tmax=0.9).get_data()
keep = np.abs(dd).max(axis=1) > speech_responsive_threshold

f, ax = plt.subplots(figsize=(8, 4))
for line, k in zip(dd, keep):
    plt.plot(line, color="r" if k else "k", alpha=0.1)

In [ ]:
for subject, epochs_i in epochs.items():
    epochs_i = epochs_i.copy().apply_baseline((-0.1, 0)).average().crop(tmin=0, tmax=0.9).get_data()
    assert epochs_i.ndim == 2
    speech_responsive_i = np.abs(epochs_i).max(axis=1) > speech_responsive_threshold

    if len(speech_responsive_i) > len(electrode_df.loc[subject]):
        speech_responsive_i = speech_responsive_i[:len(electrode_df.loc[subject])]
    electrode_df.loc[subject, "speech_responsive"] = speech_responsive_i

In [ ]:
electrode_df = electrode_df.astype({"speech_responsive": bool})

In [ ]:
electrode_df.to_csv(f"{outdir}/electrode_df.csv")

## Model sketch

In [ ]:
import json
with open("causal_candidate_populations.json", "r") as f:
    populations = json.load(f)

In [ ]:
populations

In [ ]:
from tqdm.auto import trange

all_results = []
all_results_meta = {}
populations_dict = {}

# NB, with split_strategy="extreme", what's being randomized across runs
# is only the shuffling in the nested CV splits for estimating phase 1/2
# encoders.
n_repeats = 10
split_strategy = "extreme" # "stratify" or "extreme"

for repeat in trange(n_repeats):
    for population in tqdm(populations, leave=False):
        result = run_causal4_analysis(
            epochs, subject=population["subject"],
            phoneme_pair=population["phoneme_pair"],
            population_A=population["electrodes_a"],
            population_A_window=population["window_a"],
            split_strategy=split_strategy,
        )

        key = (population["subject"], population["phoneme_pair"],
               str(population["cluster_a"]))
        populations_dict[key] = population

        repeat_key = (key, repeat)
        all_results_meta[repeat_key] = result

In [ ]:
decoder_results = pd.DataFrame([
    {
        "name": "_".join(key[0]),
        "repeat": key[1],
        "subject": result.subject,
        "population_A": result.population_A,
        "population_A_window": result.population_A_window,

        "roc_auc_val": result.phase1_val_score,
    }
    for key, result in all_results_meta.items()
])

In [ ]:
decoder_results.groupby("name").roc_auc_val.agg(["mean", "std"]).sort_values("mean")

In [ ]:
order = decoder_results.groupby("name").roc_auc_val.agg(["mean", "std"]).sort_values("mean").index
g = sns.catplot(data=decoder_results.reset_index(), x="roc_auc_val", y="name", order=order, kind="box", aspect=0.5, height=5)
g.axes.flat[0].axvline(0.5, color="k", linestyle="--")
None

In [ ]:
# study relationship between left phoneme probability and resampled value for each population A
calibration_df = []
for ((subject, phoneme_pair, population_A), repeat), result in all_results_meta.items():
    calibration_df.append(
        pd.concat([pd.Series(result.p_left_phoneme, name="p_left_phoneme"),
                   result.test_trial_metadata.reset_index()[["index", "resampled", "behavior_categorical", "label_behavior"]]], axis=1).assign(
                   subject=subject, phoneme_pair=phoneme_pair, population_A=population_A, repeat=repeat
                   ))
calibration_df = pd.concat(calibration_df)
calibration_df["site"] = calibration_df.subject + " " + calibration_df.phoneme_pair + " " + calibration_df.population_A

In [ ]:
sns.catplot(data=calibration_df, x="resampled", y="p_left_phoneme",
            col="site", col_wrap=3, kind="violin", height=3, aspect=1.5)

In [ ]:
sns.catplot(data=calibration_df.groupby(["site", "subject", "phoneme_pair", "population_A", "resampled", "index"])[["behavior_categorical", "p_left_phoneme"]].mean().reset_index(),
            x="behavior_categorical", y="p_left_phoneme",
            col="site", col_wrap=3, kind="violin", height=3, aspect=1.5)

In [ ]:
calibration_df.to_csv(f"{outdir}/calibration_df.csv", index=False)

## Start B searchlight

In [ ]:
searchlight_results = {}
searchlight_window_size = 0.3
searchlight_window_start = 0.5
searchlight_window_end = 2.5

searchlight_method = "spearmanr"  # ttest, spearmanr, or pearsonr

# pre-compute p(gt phoneme) for all experiments
searchlight_decoder_outputs = {}
for experiment in populations_dict.keys():
    subject, phoneme_pair, population_A = experiment
    
    p_gt_phoneme = np.array([all_results_meta[(experiment, repeat)].p_gt_phoneme
                             for repeat in range(n_repeats)]).T
    p_gt_phoneme_mean = p_gt_phoneme.mean(1)

    test_trial_metadata = all_results_meta[(experiment, 0)].test_trial_metadata
    mask_left = test_trial_metadata.lexical_evidence == 0

    searchlight_decoder_outputs[experiment] = {
        "p_gt_phoneme_mean": p_gt_phoneme_mean,
        "test_trial_metadata": all_results_meta[(experiment, 0)].test_trial_metadata,
        "mask_left": mask_left,
    }

searchlight_electrode_activations = {}
for experiment in tqdm(populations_dict.keys()):
    subject, phoneme_pair, population_A = experiment
    result_key = (subject, phoneme_pair, population_A)
    prev_results = all_results_meta[experiment, 0]

    population_spec = populations_dict[experiment]
    population_A_start_window, population_A_end_window = population_spec["window_a"]

    # searchlight window should start at earliest at the end of population A window.
    # override the global setting if this is not true for this population A.
    this_searchlight_window_start = max(population_A_end_window, searchlight_window_start)
    # but if we shifted, make sure it is still aligned to searchlight_window_start + n * searchlight_window_size
    if (this_searchlight_window_start - searchlight_window_start) % searchlight_window_size != 0:
        this_searchlight_window_start += searchlight_window_size - \
            ((this_searchlight_window_start - searchlight_window_start) % searchlight_window_size)

    searchlight_window_size_samp = int(searchlight_window_size * prev_results.epochs.info["sfreq"])
    searchlight_window_start_samp = prev_results.epochs.time_as_index(this_searchlight_window_start)[0]
    searchlight_window_end_samp = prev_results.epochs.time_as_index(searchlight_window_end)[0]

    # outcome from A population which will be correlated with HGA: p(gt phoneme)
    this_decoder_outputs = searchlight_decoder_outputs[experiment]
    test_trial_metadata = this_decoder_outputs["test_trial_metadata"]
    p_gt_phoneme_mean = this_decoder_outputs["p_gt_phoneme_mean"]
    mask_left = this_decoder_outputs["mask_left"]
    # control quantity: `resampled` value for the trial
    control_predictor = test_trial_metadata.resampled

    window_starts = np.arange(
        searchlight_window_start_samp,
        searchlight_window_end_samp - searchlight_window_size_samp,
        searchlight_window_size_samp
    )
    window_ends = window_starts + searchlight_window_size_samp

    if searchlight_method in ["spearmanr", "pearsonr"]:
        eval_epochs = prev_results.epochs[prev_results.test_trial_metadata.index]

        eval_epochs_left = eval_epochs[mask_left]
        eval_epochs_right = eval_epochs[~mask_left]

        eval_epochs_left = eval_epochs_left.get_data()
        eval_epochs_right = eval_epochs_right.get_data()

        for window_start, window_end in zip(window_starts, window_ends):
            # get the data for this window
            window_data_left = eval_epochs_left[:, :, window_start:window_end]
            window_data_right = eval_epochs_right[:, :, window_start:window_end]

            # average across time
            window_data_left_mean = window_data_left.mean(2)
            window_data_right_mean = window_data_right.mean(2)

            searchlight_electrode_activations[*experiment, window_start, window_end] = {
                "metadata": prev_results.test_trial_metadata,
                "mask_left": mask_left,
                "window_data_left_mean": window_data_left_mean,
                "window_data_right_mean": window_data_right_mean,
                "control_predictor": control_predictor,
            }

            results, control_results = {}, {}
            for side, window_data_mean, p_gt_phoneme_mean_side, control_predictor_side in [
                    ("left", window_data_left_mean, p_gt_phoneme_mean[mask_left], control_predictor[mask_left]),
                    ("right", window_data_right_mean, p_gt_phoneme_mean[~mask_left], control_predictor[~mask_left])]:

                if searchlight_method == "spearmanr":
                    side_test = [spearmanr(window_data_mean[:, i], p_gt_phoneme_mean_side)
                                 for i in range(window_data_mean.shape[1])]
                    corr, pval = zip(*side_test)

                    control_side_test = [spearmanr(window_data_mean[:, i], control_predictor_side)
                                         for i in range(window_data_mean.shape[1])]
                    control_corr, control_pval = zip(*control_side_test)
                elif searchlight_method == "pearsonr":
                    corr, pval = pearsonr(window_data_mean, p_gt_phoneme_mean_side[:, None], axis=0)
                    control_corr, control_pval = pearsonr(window_data_mean, control_predictor_side[:, None], axis=0)
                results[side] = (corr, pval)
                control_results[side] = (control_corr, control_pval)

            # save results
            searchlight_results[*result_key, window_start] = pd.DataFrame({
                "corr_left": results["left"][0],
                "p_val_left": results["left"][1],
                "corr_right": results["right"][0],
                "p_val_right": results["right"][1],

                "control_corr_left": control_results["left"][0],
                "control_p_val_left": control_results["left"][1],
                "control_corr_right": control_results["right"][0],
                "control_p_val_right": control_results["right"][1],

                "electrode_idx": np.arange(len(results["left"][0])),
                "electrode_in_A": np.isin(np.arange(len(results["left"][0])),
                                          population_spec["electrodes_a"]),
                "window_start_samp": window_start,
                "window_end_samp": window_end,
                "window_start": prev_results.epochs.times[window_start],
                "window_end": prev_results.epochs.times[window_end],
                "population_A": population_A,
                "phoneme_pair": phoneme_pair,
                "subject": subject,
            })

In [ ]:
searchlight_results_df = pd.concat(searchlight_results.values())
# Only retain results for which we have electrode metadata
searchlight_results_df = electrode_df.merge(searchlight_results_df, left_index=True, right_on=["subject", "electrode_idx"], how="inner")

In [ ]:
# Only retain results for speech-responsive electrodes
searchlight_results_df = searchlight_results_df[searchlight_results_df["speech_responsive"]]
# Only retain results for included ROIs
searchlight_results_df = searchlight_results_df[searchlight_results_df["roi"].isin(include_rois)]

left_is_best = searchlight_results_df[["p_val_left", "p_val_right"]].idxmin(axis=1) == "p_val_left"
searchlight_results_df["left_is_best"] = left_is_best
searchlight_results_df["p_val_min"] = searchlight_results_df[["p_val_left", "p_val_right"]].min(axis=1)
searchlight_results_df.loc[left_is_best, "control_p_val_min"] = searchlight_results_df.loc[left_is_best, "control_p_val_left"]
searchlight_results_df.loc[~left_is_best, "control_p_val_min"] = searchlight_results_df.loc[~left_is_best, "control_p_val_right"]

# Exclude results where control p-value looks anywhere near good, or where it is better than the experimental p-value
searchlight_results_df = searchlight_results_df[searchlight_results_df["control_p_val_min"] > 0.05]
searchlight_results_df = searchlight_results_df[searchlight_results_df.p_val_min < searchlight_results_df.control_p_val_min]

searchlight_results_df = searchlight_results_df.query("p_val_min < @alpha").sort_values("p_val_min")

### Decoding behavior from B

In [ ]:
searchlight_results_df["behavior_roc_auc"] = searchlight_results_df.progress_apply(
    lambda behav_row: causal4.decode_behavior(
        subject=behav_row.subject,
        phoneme_pair=behav_row.phoneme_pair,
        population_A=behav_row.population_A,
        electrode_idx=behav_row.electrode_idx,
        window_start=behav_row.window_start_samp,
        window_end=behav_row.window_end_samp,
        searchlight_electrode_activations=searchlight_electrode_activations,
        left=True
    ), axis=1)

In [ ]:
sns.displot(searchlight_results_df.behavior_roc_auc)

### Counterfactual baseline on B

In [ ]:
plot_row = searchlight_results_df.iloc[0]
plot_side = "left" if plot_row.left_is_best else "right"

orig_rval = getattr(plot_row, f"corr_{plot_side}")
print(f"Original result: r = {getattr(plot_row, f'corr_{plot_side}')}, "
      f"p = {getattr(plot_row, f'p_val_{plot_side}')}")

print("Result through counterfactual pipeline (should match):")
_, (cfac_rval, cfac_pval) = causal4.counterfactual_baseline(
    plot_row.subject, plot_row.phoneme_pair, plot_row.population_A,
    plot_row.electrode_idx,
    plot_row.window_start_samp,
    plot_row.window_end_samp,
    searchlight_decoder_outputs, searchlight_electrode_activations,
    sanity_check=True,
    left=plot_row.left_is_best
)[0]

print(f"Counterfactual result: r = {cfac_rval}, p = {cfac_pval}")
np.testing.assert_almost_equal(orig_rval, cfac_rval,
                               err_msg="Counterfactual pipeline did not return the expected result.")

In [ ]:
def counterfactual_for_row(row):
    ret = causal4.counterfactual_baseline(
        row.subject, row.phoneme_pair, row.population_A,
        row.electrode_idx, row.window_start_samp, row.window_end_samp,
        searchlight_decoder_outputs, searchlight_electrode_activations,
        sanity_check=False,
        left=row.left_is_best
    )

    if len(ret) == 0:
        # No counterfactual results, return NaNs
        return pd.Series({
            "counterfactual_spearmanr": np.nan,
            "counterfactual_spearmanr_list": [],
            "counterfactual_spearmanr_z": np.nan,
            "counterfactual_test_z": np.nan,
            "counterfactual_test_p": np.nan,
            "counterfactual_n": 0,
        })

    counterfactual_df = pd.DataFrame([
        {
            "subject": subject,
            "phoneme_pair": phoneme_pair,
            "population_A": population_A,
            "spearmanr": stat.statistic,
            "p_val": stat.pvalue,
        }
        for (subject, phoneme_pair, population_A), stat in ret
    ])
    # Fisher z-transform the Spearman correlation
    counterfactual_df["spearmanr_z"] = np.arctanh(counterfactual_df.spearmanr)

    # compute statistic on the observed Spearman correlation
    spearmanr_obs = row.corr_left if row.p_val_left < row.p_val_right else row.corr_right
    z_stat = (np.arctanh(spearmanr_obs) - counterfactual_df.spearmanr_z.mean()) / counterfactual_df.spearmanr_z.std()
    dof = len(counterfactual_df) - 1  # degrees of freedom
    p_value = 2 * stats.t.sf(np.abs(z_stat), dof)

    return pd.Series({
        "counterfactual_spearmanr": counterfactual_df.spearmanr.mean(),
        "counterfactual_spearmanr_list": counterfactual_df.spearmanr.tolist(),
        "counterfactual_spearmanr_z": counterfactual_df.spearmanr_z.mean(),

        "counterfactual_test_z": z_stat,
        "counterfactual_test_p": p_value,
        "counterfactual_n": len(counterfactual_df),
    })

In [ ]:
searchlight_results_df = pd.concat([
    searchlight_results_df,
    searchlight_results_df.progress_apply(counterfactual_for_row, axis=1)
], axis=1)

## Plot

In [ ]:
# pre-compute bounds for TIMIT plots
timit_bounds = viz.precompute_timit_bounds(timit_epoch_sources, subjects=epochs.keys())

In [ ]:
# cache z-score parameters for each subject and electrode
parameter_cache = {}

def plot_searchlight_response(row):
    subject = row.subject
    phoneme_pair = row.phoneme_pair
    population_A = row.population_A
    population_B = [row.electrode_idx]
    left = row.left_is_best
    population_B_window = (int(row.window_start_samp), int(row.window_end_samp))

    plot_key = (subject, phoneme_pair, population_A)
    plot_num_quantiles = 4
    plot_meta = all_results_meta[plot_key, 0]

    population_A_spec = populations_dict[(subject, phoneme_pair, population_A)]
    population_A_window = population_A_spec["window_a"]
    # convert to samples
    population_A_window = (
        plot_meta.epochs.time_as_index(population_A_window[0])[0],
        plot_meta.epochs.time_as_index(population_A_window[1])[0],
    )

    # sanity check: test trials are the same across repeats
    test_trial_indices = np.array([all_results_meta[(plot_key, repeat)].test_trial_metadata.index
                                    for repeat in range(n_repeats)]).T
    for i in range(1, test_trial_indices.shape[1]):
        np.testing.assert_array_equal(
            test_trial_indices[:, i],
            test_trial_indices[:, 0],
        )

    # merge estimated probabilities from repeats
    p_gt_phoneme = np.array([all_results_meta[(plot_key, repeat)].p_gt_phoneme
                            for repeat in range(n_repeats)]).T
    p_gt_phoneme_binned = pd.qcut(p_gt_phoneme.mean(1), plot_num_quantiles)

    plot_meta_df = plot_meta.test_trial_metadata.copy()
    plot_meta_df["p_gt_phoneme"] = p_gt_phoneme.mean(1)
    plot_meta_df["p_gt_phoneme_binned"] = p_gt_phoneme_binned
    plot_meta_df["p_gt_phoneme_bin_center"] = plot_meta_df.p_gt_phoneme_binned.apply(
        lambda x: x.mid
    ).astype(float).round(3)

    def get_textgrid_path(row):
        return Path(tg_dir) / (Path(row.wav_file).with_suffix(".TextGrid").name)
    plot_meta_df["textgrid_path"] = plot_meta_df.apply(get_textgrid_path, axis=1)
    plot_meta_df = plot_meta_df.rename_axis("epoch_idx").reset_index()

    # cross by electrodes
    plot_meta_df = pd.merge(plot_meta_df,
                 electrode_df.loc[plot_meta.subject].loc[population_B].reset_index(),
                 how="cross")

    ####

    g_scatter = causal4.plot_causal4_scatter(
        plot_meta, plot_meta_df, subject, population_B_window)

    ####

    # displot showing spearmanr results
    spearmanr_results = np.array(row.counterfactual_spearmanr_list)
    g_displot = sns.displot(
        spearmanr_results,
        kind="kde", fill=True, color="blue",
        height=2.5, aspect=3,
    )
    spearmanr_obs = row.corr_left if row.p_val_left < row.p_val_right else row.corr_right
    g_displot.ax.axvline(spearmanr_obs, color="red", linestyle="--", label="Observed", linewidth=2)
    g_displot.ax.set_xlabel("Spearman correlation")
    g_displot.ax.set_title(f"Permutation baseline results\n(z={row.counterfactual_test_z:.2f}, p={row.counterfactual_test_p:.2g})")

    ####

    g = causal4.plot_causal4_evoked(
        plot_meta, plot_meta_df, subject, population_A_window, population_B_window,
        hue="p_gt_phoneme_bin_center"
    )

    ####

    # plot re-aligned to behavior
    def plot_facet_evoked_align_behavior(data, color, **kwargs):
        electrode_idx = data.electrode_idx.iloc[0]
        epoch_idxs = data.epoch_idx

        word_end = data.word_end.iloc[0]
        tg_path = data.textgrid_path.iloc[0]
        tg = textgrid.TextGrid.fromFile(str(tg_path))

        ax = plt.gca()
        ax.set_xlabel("Time relative to behavior onset (sec)")
        ax.set_ylabel("HGA")
        ax.set_title(f"{subject} {electrode_idx + 1}, {word_end}")

        # plot epoched response at this electrode
        plot_epochs = plot_meta.epochs[epoch_idxs]
        plot_epochs = causal4.realign_epochs_by_behavior(plot_epochs)

        plot_epoch_data = plot_epochs.copy().pick(electrode_idx).get_data().squeeze(1)
        assert plot_epoch_data.ndim == 2  # n_trials * n_times

        plot_times = plot_epochs.times
        plot_epoch_data_mean = np.nanmean(plot_epoch_data, 0)
        plot_epoch_data_sem = np.nanstd(plot_epoch_data, 0) / np.sqrt((~np.isnan(plot_epoch_data)).sum(0))
        ax.plot(plot_times, plot_epoch_data_mean, color=color, alpha=0.5, **kwargs)
        ax.fill_between(plot_times, plot_epoch_data_mean - plot_epoch_data_sem,
                        plot_epoch_data_mean + plot_epoch_data_sem, color=color, alpha=0.2)
        
        ax.set_xlim(plot_epochs.times[0], plot_epochs.times[-1])

        return ax
    
    g_evoked_by_behavior = sns.FacetGrid(
        plot_meta_df,
        row="electrode_idx",
        hue="p_gt_phoneme_bin_center", palette="plasma",
        col="lexical_evidence",
        aspect=3, height=3, sharey="row"
    ).map_dataframe(plot_facet_evoked_align_behavior).add_legend()
    g_evoked_by_behavior.fig.suptitle("Evoked responses by P(gt phoneme), aligned to behavior onset")

    ####

    g_evoked_resampled = causal4.plot_causal4_evoked(
        plot_meta, plot_meta_df, subject, population_A_window, population_B_window,
        hue="resampled"
    )

    ####

    g_raster = causal4.plot_causal4_raster(
        plot_meta, plot_meta_df, subject, population_B_window,
        sort_by="p_gt_phoneme", parameter_cache=parameter_cache)
    
    g_raster_resampled = causal4.plot_causal4_raster(
        plot_meta, plot_meta_df, subject, population_B_window,
        sort_by="resampled", parameter_cache=parameter_cache)
    
    g_raster_behavior = causal4.plot_causal4_raster(
        plot_meta, plot_meta_df, subject, population_B_window,
        sort_by="behavior_linear", parameter_cache=parameter_cache)

    ####

    timit_fig = viz.timit_subplots(
        subject, population_B[0],
        plot_phonemes=[ph.upper() for ph in phoneme_pair],
        cell_aspect=1.5,
        epoch_sources=timit_epoch_sources,
        timit_bounds_dict=timit_bounds)
    timit_fig.suptitle(f"TIMIT responses for {subject} {population_B[0] + 1}")

    return (g_scatter, g_displot,
            g, g_evoked_by_behavior, g_evoked_resampled,
            g_raster, g_raster_resampled, g_raster_behavior,
            timit_fig)

In [ ]:
plot_row =  searchlight_results_df.iloc[3]
print(plot_row)
plot_searchlight_response(plot_row)
None

In [ ]:
decoder_results.to_csv(f"{outdir}/decoder_results.csv")

In [ ]:
searchlight_results_df.to_csv(f"{outdir}/searchlight_results.csv")

In [ ]:
torch.save({"all_results_meta": all_results_meta,
            "searchlight_results_df": searchlight_results_df,
            "searchlight_electrode_activations": searchlight_electrode_activations,
            "searchlight_decoder_outputs": searchlight_decoder_outputs,
            "populations_dict": populations_dict,
            }, f"{outdir}/searchlight_results.pth")

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

with PdfPages(f"{outdir}/searchlight_results.pdf") as pdf:
    for _, row in tqdm(searchlight_results_df.sort_values("p_val_min").iterrows(), total=len(searchlight_results_df)):
        facetgrids = plot_searchlight_response(row)
        for fg in facetgrids:
            fg.tight_layout()
            fig = fg.fig if hasattr(fg, 'fig') else fg
            pdf.savefig(fig)
            plt.close(fig)

In [ ]:
# Load results with manual annotations
searchlight_manual = pd.read_csv("causal4_searchlight_results_manual.csv", index_col=0)

In [ ]:
realign_epochs_by_behavior(epochs["EC282"])

In [ ]:
if not (searchlight_manual.index == searchlight_results_df.index).all():
    raise ValueError("Manual annotations do not match the current searchlight results. "
                     "Probably need to repeat manual annotations.")

In [ ]:
all_results_meta.keys()

In [ ]:
plot_subject = "EC260"
plot_electrode_idx = 18
plot_phoneme_pair = "dn"
plot_population_A = "5"
plot_key = (plot_subject, plot_phoneme_pair, plot_population_A)

plot_results = all_results_meta[plot_key, 0]
plot_ep = plot_results.epochs[plot_results.test_trial_metadata.index]
p_gt_phoneme = np.array([all_results_meta[(plot_key, repeat)].p_gt_phoneme
                            for repeat in range(n_repeats)]).T
p_gt_phoneme = p_gt_phoneme.mean(1)

rts = plot_ep.metadata["slider.rt"].combine(plot_ep.tmax, min)
plot_ep.plot_image(
    picks=[plot_electrode_idx],
    # overlay_times=rts,
    order=p_gt_phoneme.argsort())

In [ ]:
prev_results.epochs.plot_image(picks=[])

In [ ]:
searchlight_manual

## Cross-reference with EOIs

In [ ]:
searchlight_results_df.head()[["subject", "electrode_idx"]]

In [ ]:
eoi_df = pd.read_csv("outputs/trf_eois/eois.csv")

In [ ]:
eoi_df

In [ ]:
eoi_df_wide = eoi_df.rename(columns={"electrode": "electrode_idx"}).pivot_table(index=["subject", "electrode_idx"], columns="feature_block", values="unique_variance")
eoi_df_wide

In [ ]:
mrg = pd.merge(searchlight_results_df, eoi_df_wide, left_on=["subject", "electrode_idx"], right_index=True, how="left")
mrg

In [ ]:
mrg.loc[mrg.p_val_min == mrg.p_val_left, "corr_best"] = mrg.loc[mrg.p_val_min == mrg.p_val_left, "corr_left"].abs()
mrg.loc[mrg.p_val_min == mrg.p_val_right, "corr_best"] = mrg.loc[mrg.p_val_min == mrg.p_val_right, "corr_right"].abs()
mrg["log_p_val_min"] = np.log10(mrg.p_val_min)

In [ ]:
sns.regplot(data=mrg.reset_index(), x="log_p_val_min", y="mismatch")

## Cross-reference with single electrode decoding

In [ ]:
dec_df = pd.read_csv("outputs/single_electrode_decoding/10/acoustic/scores.csv")

In [ ]:
dec_summary = dec_df.groupby(["subject", "electrode_idx", "phoneme_pair"]).roc_auc.max()

In [ ]:
mrg = pd.merge(searchlight_results_df,
         dec_summary.rename("lexical_evidence_roc_auc"),
         left_on=["subject", "electrode_idx", "phoneme_pair"],
         right_index=True, how="left")

mrg["log_p_val_min"] = -np.log10(mrg.p_val_min)

In [ ]:
sns.regplot(data=mrg.reset_index(), x="log_p_val_min", y="lexical_evidence_roc_auc",)